# TP 4 - Algoritmos Genéticos

## Ejercicio 5

Resolver el problema de las 8 reinas de ajedrez con un AGS (Referencia sección 4.3 del libro de Russell y Norvig), Max Bezzel en 1948 inventó el problema de las 8 reinas que consiste en ubicar en un tablero de ajedrez 8 reinas que no se amenacen, una reina está amenazada si existe en su misma fila, columna o diagonal otra reina. Considere un tablero de 8×8 celdas como el de la figura más abajo, para resolverlo con un AG deberá definir:
1. la codificación de los individuos
2. la función de selección natural
3. la función de combinación
4. la función de mutación 

Resuélvalo también con Búsqueda Tabú especificando: 
1. la codificación de la solución, 
2. la función objetivo, 
3. la función de vecindario 
4. la lista y los movimientos tabú


## 1. Imports y función de evaluación

Función objetivo: calacular cuántos pares de reinas se están amenazando en el tablero actual. Una reina amenaza a otra si comparten la misma fila o la misma iagonal (las columnas ya estan protegidas por la forma en que se codifica). 
El objetivo es que los ataques lleguen a cero.

In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

def calcular_ataques(estado):
    """
    estado: lista de 8 enteros (fila de la reina en cada columna).
    Max ataques posibles: 28. Min ataques: 0 (Solución perfecta).
    """
    ataques = 0
    n = len(estado)
    for i in range(n):
        for j in range(i + 1, n):
            # Se amenazan si están en la misma fila o diagonal
            if estado[i] == estado[j] or abs(estado[i] - estado[j]) == abs(i - j):
                ataques += 1
    return ataques

## 2. Algoritmo Genético Simple (AGS)

### Codificación de los individuos

Representación basada en valores enteros (vectores de posición), un individuo (cromosoma) es una lista de 8 posiciones
1. El índice del arreglo (0 a 7) representa la columna del tablero
2. El valor de ese índice (0 a 7) representa la fila en la que se ubica la reina. Esta codificación evita automáticamente que dos reinas se coloquen en la misma columna, reduciendo el espacio de búsqueda 

In [2]:
def generar_poblacion_inicial(pop_size):
    """Genera la población inicial con individuos aleatorios de 8 genes"""
    return [np.random.randint(0, 8, 8).tolist() for _ in range(pop_size)]

### Función de selección natural

Utilizamos el método de selección por torneo: Se toman 3 individuos al azar de la población y se evalúa su aptitud, donde el individuo con el menor número de ataques gana el torneo y es seleccionado para reproducirse

In [3]:
def seleccion_torneo(pop, fitness_pop, tourn_size=3):
    participantes = random.sample(range(len(pop)), tourn_size)
    mejor_idx = min(participantes, key=lambda x: fitness_pop[x]) # Gana el que tiene MENOR cantidad de ataques
    return pop[mejor_idx]

### Función de combinación

Aplicamos el cruce por un solo punto: Se selecciona aleatoriamente un punto de corte a lo largo del cromosoma. El "hijo" hereda los genes del padre 1 desde el inicio hasta el punto de corte, y los genes del padre 2 desde el punto de corte hasta el final de la lista.

In [4]:
def cruce_un_punto(p1, p2):
    pt = random.randint(1, 6)  # Punto de corte (pt) aleatorio entre 1 y 6 (para no copiar idéntico a un padre)
    hijo = p1[:pt] + p2[pt:]
    return hijo

### Función de mutación

Se implementa la mutación uniforme discreta con una probabilidad determinada (en este caso del 20%) en el hijo recién creado. Se elige una columna completamente al azar y se le asigna una nueva fila aleatoria. Esto simula mover una reina para inyectar diversidad y evitar estancamientos en óptimos locales.

In [5]:
def mutar(hijo, mut_rate=0.2):
    if random.random() < mut_rate:
        col_mutar = random.randint(0, 7)
        fila_nueva = random.randint(0, 7)
        hijo[col_mutar] = fila_nueva
    return hijo

### Bucle principal del AGS

Donde se unen los componentes anteriores en el ciclo evolutivo. La población evoluciona generación tras generación hasta encontrar una solución con cero ataques o hasta alcanzar el límite de generaciones.

In [6]:
def ags_8reinas(pop_size=100, generaciones=1000):
    pop = generar_poblacion_inicial(pop_size)
    historial_estados = []
    
    for gen in range(generaciones):
        fitness_pop = [calcular_ataques(ind) for ind in pop]
        
        # Guardar el mejor de esta generación para el historial
        mejor_idx_gen = np.argmin(fitness_pop)
        mejor_ind_gen = pop[mejor_idx_gen]
        mejor_fitness = fitness_pop[mejor_idx_gen]
        historial_estados.append(mejor_ind_gen.copy())
        
        # Condición de corte: Solución perfecta
        if mejor_fitness == 0:
            print(f"AGS encontró la solución en la generación {gen}")
            return mejor_ind_gen, historial_estados
            
        # Crear nueva generación
        nueva_pop = []
        for _ in range(pop_size):
            p1 = seleccion_torneo(pop, fitness_pop)
            p2 = seleccion_torneo(pop, fitness_pop)
            hijo = cruce_un_punto(p1, p2)
            hijo = mutar(hijo)
            nueva_pop.append(hijo)
            
        pop = nueva_pop
        
    print("AGS finalizó sin solución perfecta (límite de generaciones).")
    return mejor_ind_gen, historial_estados

## 3. Búsqueda Tabú (BT)

### Codificación de la solución y función objetivo

Ambos son iguales al algoritmo genético simple.

In [7]:
def inicializar_solucion_tabu():
    return np.random.randint(0, 8, 8).tolist()

### Función de vecindario y movimientos Tabú

El vecindario se conforma por todos los tableros que se pueden generar moviendo una única reina a una fila diferente dentro de su misma columna (56 posibles vecinos).
Los movimientos tabú se registran como la tupla (columna , nueva_fila). Si movemos una reina, ese movimiento entra a la lista tabú y queda prohibido realizarlo durante una cantidad específica de iteraciones, evitando asi que el algoritmo entre en un bucle infinito deshaciendo y rehaciendo el mismo movimiento.

In [8]:
def busqueda_tabu_8reinas(max_iter=1000, tabu_tenure=12):
    estado_actual = inicializar_solucion_tabu()
    mejor_estado_global = estado_actual.copy()
    mejor_ataques_global = calcular_ataques(estado_actual)
    
    lista_tabu = {} # Diccionario para llevar el tiempo de vida del tabú
    historial_estados = [estado_actual.copy()]
    
    for iteracion in range(max_iter):
        if mejor_ataques_global == 0:
            print(f"Tabú encontró la solución en la iteración {iteracion}")
            break

        mejor_vecino = None
        mejor_ataque_vecino = float('inf')
        movimiento_elegido = None
        
        for col in range(8):
            for fila in range(8):
                if estado_actual[col] != fila:
                    vecino = estado_actual.copy()
                    vecino[col] = fila
                    ataques_vecino = calcular_ataques(vecino)
                    
                    movimiento = (col, fila)
                    es_tabu = lista_tabu.get(movimiento, 0) > iteracion # Excepción por aspiración: Si es tabú pero mejora el óptimo global, lo permitimos
                    
                    if (not es_tabu) or (ataques_vecino < mejor_ataques_global):
                        if ataques_vecino < mejor_ataque_vecino:
                            mejor_ataque_vecino = ataques_vecino
                            mejor_vecino = vecino
                            movimiento_elegido = movimiento
                            
        if mejor_vecino is not None:
            estado_actual = mejor_vecino
            lista_tabu[movimiento_elegido] = iteracion + tabu_tenure
            
            if mejor_ataque_vecino < mejor_ataques_global:
                mejor_ataques_global = mejor_ataque_vecino
                mejor_estado_global = estado_actual.copy()
                
        historial_estados.append(estado_actual.copy())
        
    return mejor_estado_global, historial_estados

## Gráfica de Resultados



In [9]:
# 1. FUNCIÓN GRÁFICA

def graficar_tablero(estado, titulo, ax):
    tablero = np.zeros((8, 8))
    tablero[1::2, ::2] = 1
    tablero[::2, 1::2] = 1
    
    ax.imshow(tablero, cmap='gray_r', vmin=0, vmax=1)
    ax.set_title(titulo, fontsize=11)
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))
    ax.set_xticklabels(['A','B','C','D','E','F','G','H'])
    
    for col, fila in enumerate(estado):
        ax.text(col, fila, '♛', fontsize=24, ha='center', va='center', color='red')

# 2. BLOQUE FINAL: EJECUCIÓN Y ANIMACIÓN PASO A PASO

if __name__ == '__main__':
    print("Simulando algoritmos para recolectar los estados...")
    sol_ags, hist_est_ags = ags_8reinas()
    sol_tabu, hist_est_tabu = busqueda_tabu_8reinas()

    print(f" -> AGS generó {len(hist_est_ags)} cuadros.")
    print(f" -> TABÚ generó {len(hist_est_tabu)} cuadros.")
    print("Preparando el reproductor de video... (esto puede tardar unos segundos)")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Progreso Paso a Paso: AGS vs Búsqueda Tabú', fontsize=16, fontweight='bold')
    
    total_frames = max(len(hist_est_ags), len(hist_est_tabu))
    
    def update(frame):
        ax1.clear()
        ax2.clear()
        
        # --- Lógica para AGS ---
        idx_ags = min(frame, len(hist_est_ags) - 1)
        estado_ags = hist_est_ags[idx_ags]
        amenazas_ags = calcular_ataques(estado_ags)
        graficar_tablero(estado_ags, f'AGS\nGen: {idx_ags} | Amenazas: {amenazas_ags}', ax1)
        
        # --- Lógica para TABÚ ---
        idx_tabu = min(frame, len(hist_est_tabu) - 1)
        estado_tabu = hist_est_tabu[idx_tabu]
        amenazas_tabu = calcular_ataques(estado_tabu)
        graficar_tablero(estado_tabu, f'Búsqueda Tabú\nIter: {idx_tabu} | Amenazas: {amenazas_tabu}', ax2)
        
    ani = animation.FuncAnimation(fig, update, frames=total_frames, interval=200, repeat=False)
    plt.close() # Cerramos el plot para evitar imagen estática duplicada
    display(HTML(ani.to_jshtml())) # Mostramos el reproductor interactivo

Simulando algoritmos para recolectar los estados...
AGS encontró la solución en la generación 15
Tabú encontró la solución en la iteración 12
 -> AGS generó 16 cuadros.
 -> TABÚ generó 13 cuadros.
Preparando el reproductor de video... (esto puede tardar unos segundos)
